In [12]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"

# datasets
from lerobot.datasets.utils import hw_to_dataset_features
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# policy
from lerobot.policies.act.modeling_act import ACTPolicy

# robots
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.cameras.configs import Cv2Rotation
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# record utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say, init_logging
from lerobot.utils.visualization_utils import _init_rerun
from lerobot.record import record_loop

# torch
from torch import cuda

# utils
from dotenv import load_dotenv
import time

# my code
from src.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR, EVAL_DIR, OUTPUTS_DIR
from src.const import TABLE_START_POSE, TABLE_START_POSE_OPEN
from src.robot_utils import move_robot_to_pose
from src.utils import write_eval_stats

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

Using device: cuda
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Set Params:

In [2]:
PUSH_TO_HUB     = False
SAVE_TO_DATASET = True
REPO_NAME         = 'so101_leg_pick_and_place'
EXPERIMENT_NAME   = '50_episodes_v2'
POLICY_CHECKPOINT = '100000'
POLICY_TYPE = 'act'
NUM_EPISODES = 5
FPS = 30
EPISODE_TIME_SEC = 60
RESET_TIME_SEC = 10
EVAL_TYPE = 'real'

Log to HF if pushing:

In [3]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

Initialize Policy:

In [4]:
# Initialize the policy
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / POLICY_CHECKPOINT / 'pretrained_model'
policy = ACTPolicy.from_pretrained(pretrained_name_or_path = policy_path, local_files_only=True)

Loading weights from local directory


Robots Config

In [5]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=0, width=640, height=480, fps=30),
    "top_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30, rotation=Cv2Rotation.NO_ROTATION),
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)

robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

Build Dataset:

In [7]:
# configure the dataset features
action_features = hw_to_dataset_features(robot.action_features, "action")
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}

if REPO_NAME is not None:
    # create dataset
    policy_eval_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"
    kwargs = {
    "repo_id": f"{HF_NAME}/{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    "root"   : str(policy_eval_path)
    }
    
    if policy_eval_path.exists():
        dataset=LeRobotDataset(**kwargs)
    else:
        dataset = LeRobotDataset.create(
            **kwargs,
            fps=FPS,
            features=dataset_features,
            robot_type=robot.name,
            use_videos=True,
            image_writer_threads=4,
        )
else:
    dataset = None

In [8]:
# Initialize the keyboard listener and rerun visualization
_, events = init_keyboard_listener()
_init_rerun(session_name="recording")

# Connect the robot
robot.connect()
# teleop.connect()

Initialize loop kwargs:

In [9]:
kwargs = {
    "robot": robot,
    "events": events,
    "fps": FPS,
    "policy": policy,
    "display_data": True,
    "teleop": teleop
}

# if SAVE_TO_DATASET:
kwargs["dataset"] = dataset
kwargs["single_task"] = REPO_NAME

#### Recording loop:

In [ ]:
results = {"per_episode": []}
start_time = time.time()

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
init_logging(Path(f'C:/Github/IDC_Project-Sim_Real_Sim/outputs/logs/log_{ts}.txt'))

for episode_idx in range(NUM_EPISODES):
    log_say(f"Running inference, recording eval episode {episode_idx + 1} of {NUM_EPISODES}")
    move_robot_to_pose(robot, TABLE_START_POSE, 1, 30)
    record_loop(**kwargs, control_time_s=EPISODE_TIME_SEC)

    # reset / save logic
    if not events["stop_recording"] and (episode_idx < NUM_EPISODES - 1 or events["rerecord_episode"]):
        log_say("Reset the environment")
        move_robot_to_pose(robot, TABLE_START_POSE_OPEN, 1, 30)

        reset_start = time.time()
        # if SAVE_TO_DATASET:
        #     dataset.save_episode()
        elapsed = time.time() - reset_start
        remaining = RESET_TIME_SEC - elapsed
        if remaining > 0:
            time.sleep(remaining)

        move_robot_to_pose(robot, TABLE_START_POSE, 1, 30)
    # else:
    #     if SAVE_TO_DATASET:
    #         dataset.save_episode()

    dataset.clear_episode_buffer()

    # === episode logging happens at the end ===
    if events["stop_recording"]:
        log_say("ESC pressed — discarding last episode")
        move_robot_to_pose(robot, TABLE_START_POSE_OPEN, 1, 30)
        break  # skip writing stats for this episode

    ep = {k: None for k in ["episode_ix", "sum_reward", "max_reward", "success", "seed"]}
    ep["episode_ix"] = episode_idx
    log_say("Input success")
    ep["success"] = float(input(f"[Ep {episode_idx}] Enter success (0, 0.5, 1): "))
    results["per_episode"].append(ep)

move_robot_to_pose(robot, TABLE_START_POSE, 1, 30)
elapsed = time.time() - start_time
write_eval_stats(dataset, results["per_episode"], elapsed, NUM_EPISODES)


Escape key pressed. Stopping data recording...


In [11]:
# Clean up
robot.disconnect()
# teleop.disconnect()
if PUSH_TO_HUB:
    dataset.push_to_hub()